# APICE Tutorial Notebook

This notebook demonstrates APICE EEG preprocessing and segmentation workflows.

It has two goals:
1. Show high-level pipeline wrappers for multiple files.
2. Show lower-level RawAPICE and EpochsAPICE step-by-step methods.

The examples below are designed for this repository layout and use test data under `test_data/` when available.

## 1) Install and Import Dependencies

If your environment already contains these packages, you can skip the install cell.

This section imports pipeline wrappers (`run_preprocessing`, `run_segmentation`, `preprocess_initial_steps`, `preprocess_apice_default`) and low-level APICE APIs used later.

In [ ]:
# Optional: install dependencies in the active environment
# %pip install mne mne-bids tabulate

from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


import mne

from apice.pipeline import (
    preprocess_initial_steps,
    preprocess_apice_default,
    segment_default_pipeline,
)
from apice.io import load_rawapice, load_epochapice
from apice.data_structures import RawAPICE, EpochsAPICE
from apice.filter import Filter
from apice.artifacts_rejection import ArtifactsConfiguration, run_algorithms

print("Imports OK")

## 2) Configure Input and Output Directories

Set paths for input and outputs. By default, this notebook points to repository test data.

In [ ]:
# Assume notebook runs from repository root
repo_root = Path.cwd()

# Default local test-data paths
input_dir = repo_root / "test_data" / "raw"
output_dir_preproc = repo_root / "tests" / "preprocessed"
output_dir_segmented = repo_root / "tests" / "segmented"

output_dir_preproc.mkdir(parents=True, exist_ok=True)
output_dir_segmented.mkdir(parents=True, exist_ok=True)

print("input_dir:", input_dir)
print("output_dir_preproc:", output_dir_preproc)
print("output_dir_segmented:", output_dir_segmented)
print("raw test file exists:", (input_dir / "test_recording.fif").exists())

## 3) Set Preprocessing Parameters

Define shared preprocessing parameters used by wrapper and low-level examples.

In [ ]:
# Parameters matched to the test_recording.fif data
# (The FIF file already contains an embedded montage, so montage=None.)

drop_electrodes = ['E125', 'E126', 'E127', 'E128']   # specify channels to drop a priori (e.g., outer ring channels)
reference_channels = ['VREF']                        # test recording  includes the reference channel; indicate it so it is not treated as bad channel
picks = "eeg"
crop_times = None
crop_from_beginnning = None
crop_from_end = None
resample_freq = None
stim_channels_to_annotations = False   # test recording does not use STIMs for events

# montage=None because test_recording.fif already has the montage stored inside.
# For external files (e.g. .vhdr) without an embedded montage, provide a path:
#   montage = Path(repo_root) / "electrode_layout" / "GSN-HydroCel-129.sfp")
montage = None

l_freq = 0.10
h_freq = 40
l_trans_bandwidth = 0.1
h_trans_bandwidth = 10

print("drop_electrodes:", drop_electrodes)
print("reference_channels:", reference_channels)
print("montage:", montage)

## 4) Configure Artifact Detection

Use `None` to load APICE packaged defaults from `apice/default_cfg`.

You can also pass dictionary configs or JSON paths.

In [ ]:
cfg_bad_channels_detection = None
cfg_glitches_detection = None
cfg_target_pca = None
cfg_artifacts_detection = None
cfg_spline_segments = None
cfg_spline_channels = None

# Segmentation-side default cfgs
cfg_define_bcbt_epochs = None
cfg_bad_epochs = None

print("All cfg_* are set to None (APICE defaults).")

## 5) Run Preprocessing Pipeline

This section demonstrates both:
- single-file wrappers (`preprocess_initial_steps`, `preprocess_apice_default`)
- batch wrapper (`run_preprocessing`) for multiple files.

`data_selection_method='all'` processes all eligible input files.
`data_selection_method='new'` processes only files not already present in output.

In [ ]:
# Single-file wrapper demonstration
single_raw_path = input_dir / "test_recording-raw.fif"

if single_raw_path.exists():
    raw_single = mne.io.read_raw(single_raw_path, preload=False, verbose=False)

    raw_single = preprocess_initial_steps(
        raw_single,
        drop_electrodes=drop_electrodes,
        picks=picks,
        crop_times=crop_times,
        crop_from_beginnning=crop_from_beginnning,
        crop_from_end=crop_from_end,
        resample_freq=resample_freq,
        stim_channels_to_annotations=stim_channels_to_annotations,
        montage=montage,
    )

    raw_apice_single, summary_single, report_single = preprocess_apice_default(
        raw_single,
        output_dir=output_dir_preproc,
        file_name=single_raw_path.stem,
        create_report=True,
        save_log=False,
        save_data=True,
        save_report=True,
        save_summary=True,
        save_cfg=True,
        reference_channels=reference_channels,
        l_freq=l_freq,
        h_freq=h_freq,
        l_trans_bandwidth=l_trans_bandwidth,
        h_trans_bandwidth=h_trans_bandwidth,
        cfg_bad_channels_detection=cfg_bad_channels_detection,
        cfg_glitches_detection=cfg_glitches_detection,
        cfg_target_pca=cfg_target_pca,
        cfg_artifacts_detection=cfg_artifacts_detection,
        cfg_spline_segments=cfg_spline_segments,
        cfg_spline_channels=cfg_spline_channels,
        n_jobs=-1,
    )
    print("Single-file preprocessing finished.")
else:
    print(f"Raw test file not found: {single_raw_path}")

## 6) Run Segmentation Pipeline

Define event extraction settings and run segmentation wrappers over preprocessed files.

In [ ]:
l_freq_epochs = 0.2
h_freq_epochs = 20

kwargs_events_from_annotations_for_segmentation = {
    "regexp": 'shape*',  
    "verbose": False,
}
event_time_window = (-0.2, 1.0)
baseline = (None, 0)  # use pre-stimulus period for baseline correction; set to None to skip baseline correction
evoked_by = ["shape"]

epochs_apice, evokeds_apice, summary_segmentation, report_segmentation = segment_default_pipeline(
    raw_apice_single, 
    kwargs_events_from_annotations_for_segmentation, 
    event_time_window,
    file_name="test_recording-raw-preproc",
    l_freq=l_freq_epochs,
    h_freq=h_freq_epochs,
    l_trans_bandwidth=l_trans_bandwidth,
    h_trans_bandwidth=h_trans_bandwidth,
    baseline=baseline, 
    kwargs_events_from_annotations_for_metadata=None,
    kwargs_make_metadata=None,                             
    evoked_by=evoked_by,
    output_dir=output_dir_segmented,
    save_log=False,
    save_epochs=True,
    save_only_good_epochs=False,
    save_evoked=True,
    save_report=True,
    save_summary=True,
    save_cfg=True,
    set_reference={'ref_channels':'average'},
    cfg_define_bcbt_epochs=cfg_define_bcbt_epochs,
    cfg_spline_channels=cfg_spline_channels,  
    cfg_bad_epochs=cfg_bad_epochs,              
    n_jobs=-1,
    )


print("run_segmentation completed")

## 7) Detailed Single-File Workflow with RawAPICE and EpochsAPICE

This section details wrapper methods and export/reload flow.

Important:
- After `correct_target_pca`, apply a high-pass filter.
- After `correct_spline_segments`, apply a high-pass filter.

This mirrors the order used inside `preprocess_apice_default`.


### Aplly APICE preprocessing steps using default configurations parameters

In [ ]:
# Load raw data
raw_source_file = input_dir / "test_recording-raw.fif"
raw_base = mne.io.read_raw(raw_source_file, preload=False, verbose=False)

# Apply initial preprocessing steps (e.g., drop channels, crop, resample, set montage) to the raw data
raw_base = preprocess_initial_steps(raw_base, 
                                    drop_electrodes=drop_electrodes,
                                    picks=picks,
                                    crop_times=crop_times,
                                    crop_from_beginnning=crop_from_beginnning,
                                    crop_from_end=crop_from_end,
                                    resample_freq=resample_freq,
                                    stim_channels_to_annotations=stim_channels_to_annotations,
                                    montage=montage,
                                    )

# Band pass filter the data to remove drifts and high-frequency noise before artifact detection and correction steps
Filter(raw_base, l_freq=l_freq, h_freq=h_freq, l_trans_bandwidth=l_trans_bandwidth, h_trans_bandwidth=h_trans_bandwidth, n_jobs=-1)

# Initialize RawAPICE from the preprocessed raw data
raw_demo = RawAPICE(raw_base)

# Apply artifact detection algorithm targeting bad channels
raw_demo.detect_bad_channels()

# Apply artifact detection algorithm targeting glitches
raw_demo.detect_glitches()

# Correct glitches using target PCA-based correction
raw_demo.correct_target_pca()
Filter(raw_demo, l_freq=l_freq, h_freq=None, l_trans_bandwidth=l_trans_bandwidth, h_trans_bandwidth=h_trans_bandwidth, n_jobs=-1)

# Apply artifact detection algorithm targeting general artifacts (e.g., motion artifacts)
raw_demo.detect_artifacts()

# Correct artifacts using spline interpolation of segments
raw_demo.correct_spline_segments()
Filter(raw_demo, l_freq=l_freq, h_freq=None, l_trans_bandwidth=l_trans_bandwidth, h_trans_bandwidth=h_trans_bandwidth, n_jobs=-1)

# Interpolate bad channels using spline interpolation across channels
raw_demo.correct_spline_channels()

# Apply artifact detection algorithm targeting general artifacts again after corrections
raw_demo.detect_artifacts()

# Visualize the artifact structure of the raw data after all detections and corrections
raw_demo.plot_artifact_structure(artifact='all')

# Export and reload raw
raw_export_name = "tutorial_raw"
raw_demo.export(raw_export_name, output_dir_preproc, data_suffix="-preproc")
raw_reloaded = load_rawapice(output_dir_preproc / f"{raw_export_name}-preproc.fif")
print("Reloaded raw type:", type(raw_reloaded))


### Apply APICE segmentation steps using default configurations

In [ ]:
# Segment continuous data, run epoch-level correction and rejection, then export/reload epochs
events, event_id = mne.events_from_annotations(raw_demo)
if len(events) > 0:
    epochs_demo = raw_demo.segment_continuous_data(
        events=events,
        event_id=event_id,
        epoching_kwargs=dict(tmin=-0.2, tmax=0.8, baseline=None, preload=True, reject_by_annotation=False),
    )
    print("is EpochsAPICE:", isinstance(epochs_demo, EpochsAPICE))

    # Update artifact parameters for the segmented data (optional, but recommended to ensure appropriate parameters for epoch-level artifact handling, otherwise defaults are used)
    if cfg_define_bcbt_epochs is not None:
        epochs_demo.update_artifacts_params(**cfg_define_bcbt_epochs)
    
    # Define BadTimes and BadChannels for the segmented data
    epochs_demo.define_bcbt()

    # interpolate bad channels per epochs using spline interpolation
    epochs_demo.correct_spline_channels()

    # define bad epochs based on the defined bad times and channels, as well as distance and GFP criteria, then remove them from the dataset
    epochs_demo.define_bad_epochs(bad_data=1, bad_time=0, bad_channel=0.3, lim_dist=2, lim_gfp=2)
    epochs_demo.remove_bad_epochs()

    # Save the epochs
    epochs_export_name = "tutorial_epochs-epo.fif"
    epochs_demo.export(epochs_export_name, output_dir_segmented)

    # Reload the epochs to verify export worked correctly
    epochs_reloaded = load_epochapice(output_dir_segmented / epochs_export_name)
    print("Reloaded epochs type:", type(epochs_reloaded))


## 8) Use custom configurations

Some examples are provided on how to run preprocessing steps with custom parameters

#### Create a custom configuration to detect artifacts and run the detection algorithms

In [ ]:
# Initialize a new ArtifactsConfiguration object to define the artifact detection algorithms and parameters for the segmentation step
cfg_obj = ArtifactsConfiguration()

# Add a group of algorithms. 
# It will run up to 3 loops of artifact detection and rejection, with a minimum rejection rate of 0.1% to continue looping. 
cfg_obj.add_algorithm_group('artifacts_amplitude', max_loops=3, min_rejection=0.1, define_bcbt=True)

# Add an algorithm to the group: Amplitude-based artifact detection with specified parameters
cfg_obj.add_algorithm('artifacts_amplitude', 'Amplitude', {
    'bad_data': None,
    'do_reference_data': False,
    'do_zscore': False,
    'thresh_type': 'outliers_per_channel',
    'thresh': [-2, 2],
    'mask': 0,
    'remove_bct': True,
    'remove_bt': True,
    'remove_bc': True,
}, algorithm_name='ArtifactsAmplitude')

# Add another group of algorithms
# It will run up to 3 loops of artifact detection and rejection, with a minimum rejection rate of 0.1% to continue looping.
cfg_obj.add_algorithm_group('artifacts_maxchange500', max_loops=3, min_rejection=0.1, define_bcbt=True)

# Add an algorithm to the group: MaxChange-based artifact detection with specified parameters
cfg_obj.add_algorithm('artifacts_maxchange500', 'MaxChange', {
    'bad_data': None,
    'do_reference_data': False,
    'do_zscore': False,
    'thresh_type': 'outliers_per_channel',
    'thresh': [None, 2.0],
    'time_window': 0.500,
    'time_window_step': 0.100,
    'mask': 0,
    'remove_bct': True,
    'remove_bt': True,
    'remove_bc': True,
}, algorithm_name='MaxChange500ms')

# Add another group of algorithms to modify the rejection matrix.
cfg_obj.add_algorithm_group('artifacts_modify', max_loops=1, min_rejection=0, define_bcbt=True)

# Add algorithms to the group: Short segment-based modifications with specified parameters
cfg_obj.add_algorithm('artifacts_modify', 'ShortBadSegments', {
    'time_limit': 0.050,
}, algorithm_name='ShortBadSegments', post_detection=True)
cfg_obj.add_algorithm('artifacts_modify', 'ShortGoodSegments', {
    'time_limit': 0.500,
}, algorithm_name='ShortGoodSegments', post_detection=True)

# Check the configuration
cfg_obj.check_configuration()

# save the configuration to a JSON file
output_dir_cfg = repo_root / "tests" / "cfgs"
output_dir_cfg.mkdir(parents=True, exist_ok=True)
custom_cfg_path = output_dir_cfg / "tutorial_custom_algorithms.json"
cfg_obj.save_to_json(custom_cfg_path)



In [ ]:
# Run it passing the configuration dictionry defined above.
raw_cfg = raw_demo.copy()
run_algorithms(raw_cfg, cfg_obj.cfg)

# Run providing the JSON file path instead of a dictionary with the configuration
raw_cfg = raw_demo.copy()
run_algorithms(raw_cfg, custom_cfg_path)

#### Run correction using spline interpolation with custom parameters

In [ ]:
# Define a custom configuration for the spline segments correction algorithm, which will be used in the segmentation step. 
spline_segments_cgf = {}
spline_segments_cgf['p'] = 0.5
spline_segments_cgf['p_neighbors'] = 1
spline_segments_cgf['min_good_time'] = 1.00 
spline_segments_cgf['min_intertime'] = 0.050
spline_segments_cgf['mask_time'] = 0.100
spline_segments_cgf['min_segment_time'] = 0.250
spline_segments_cgf['splice_method'] = 1
spline_segments_cgf['parallelize_mode'] = 'auto'
spline_segments_cgf['save_corrected'] = True

# Save to JSON file
custom_cfg_path = output_dir_cfg / "custom_spline_segments_cfg.json"
with open(custom_cfg_path, 'w') as f:
    json.dump(spline_segments_cgf, f, indent=4)

# Run it passing the configuration dictionry defined above.
raw_cfg.correct_spline_segments(spline_segments_cgf)

# Run providing the JSON file path for the spline segments configuration
raw_cfg.correct_spline_segments(custom_cfg_path)

